# 03 — PySpark Native Bucketing Demo

Goal: pull raw telemetry rows from Postgres, then do bucketing/window logic in Spark DataFrames.


## Imports and Spark Session


In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
spark = (
    SparkSession.builder
    .appName('pyspark-native-bucketing-demo')
    .config('spark.sql.shuffle.partitions', '8')
    .config('spark.driver.host', '127.0.0.1')
    .config('spark.driver.bindAddress', '127.0.0.1')
    .config('spark.jars.packages', 'org.postgresql:postgresql:42.7.4')
    .getOrCreate()
)
print('Spark version:', spark.version)


:: loading settings :: url = jar:file:/usr/local/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.postgresql#postgresql added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-148067a8-2091-41b6-b531-7467e527b1b5;1.0
	confs: [default]
	found org.postgresql#postgresql;42.7.4 in central
	found org.checkerframework#checker-qual;3.42.0 in central
:: resolution report :: resolve 200ms :: artifacts dl 5ms
	:: modules in use:
	org.checkerframework#checker-qual;3.42.0 from central in [default]
	org.postgresql#postgresql;42.7.4 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   2   |   0   |   0   |   0   ||   2   |   0   |
	---------------------------------------------------------

Spark version: 3.5.4


## Connection settings


In [2]:
DB_HOST='host.docker.internal'
DB_PORT='5432'
DB_NAME='observability'
DB_USER='obs_user'
DB_PASS='obs_pass'
JDBC_URL=f'jdbc:postgresql://{DB_HOST}:{DB_PORT}/{DB_NAME}'


## Load raw rows only (minimal SQL pushdown)


In [3]:
query = '''(
SELECT sampled_at, host, cpu_pct, mem_pct, region, env
FROM lab.telemetry_cpu_raw
WHERE sampled_at >= now() - interval '14 days'
) t'''
df = (spark.read.format('jdbc').option('url', JDBC_URL).option('dbtable', query).option('user', DB_USER).option('password', DB_PASS).option('driver', 'org.postgresql.Driver').load())
df.printSchema()
df.show(5, truncate=False)


root
 |-- sampled_at: timestamp (nullable = true)
 |-- host: string (nullable = true)
 |-- cpu_pct: decimal(5,2) (nullable = true)
 |-- mem_pct: decimal(5,2) (nullable = true)
 |-- region: string (nullable = true)
 |-- env: string (nullable = true)



+--------------------------+------+-------+-------+---------+-----+
|sampled_at                |host  |cpu_pct|mem_pct|region   |env  |
+--------------------------+------+-------+-------+---------+-----+
|2026-04-30 09:32:10.464532|host09|43.53  |84.80  |eu-west-1|prod |
|2026-04-30 09:32:14.689683|host03|32.84  |65.53  |us-west-2|stage|
|2026-04-30 09:32:30.981253|host25|50.46  |74.62  |us-west-2|stage|
|2026-04-30 09:32:35.464672|host45|79.75  |60.23  |us-east-1|prod |
|2026-04-30 09:32:39.056231|host67|42.63  |75.20  |eu-west-1|stage|
+--------------------------+------+-------+-------+---------+-----+
only showing top 5 rows



## Spark bucketing columns


In [4]:
df_b = (df.withColumn('hour_bucket', F.date_trunc('hour', F.col('sampled_at'))).withColumn('day_bucket', F.to_date(F.col('sampled_at'))))
df_b.select('sampled_at','hour_bucket','day_bucket').show(5, truncate=False)


+--------------------------+-------------------+----------+
|sampled_at                |hour_bucket        |day_bucket|
+--------------------------+-------------------+----------+
|2026-04-30 09:32:10.464532|2026-04-30 09:00:00|2026-04-30|
|2026-04-30 09:32:14.689683|2026-04-30 09:00:00|2026-04-30|
|2026-04-30 09:32:30.981253|2026-04-30 09:00:00|2026-04-30|
|2026-04-30 09:32:35.464672|2026-04-30 09:00:00|2026-04-30|
|2026-04-30 09:32:39.056231|2026-04-30 09:00:00|2026-04-30|
+--------------------------+-------------------+----------+
only showing top 5 rows



## Spark hourly aggregation


In [5]:
hourly = (df_b.where(F.col('env')=='prod').groupBy('hour_bucket','host').agg(F.round(F.avg('cpu_pct'),2).alias('avg_cpu'), F.max('cpu_pct').alias('peak_cpu')).orderBy(F.col('hour_bucket').desc(), F.col('host')))
hourly.show(20, truncate=False)


+-------------------+------+-------+--------+
|hour_bucket        |host  |avg_cpu|peak_cpu|
+-------------------+------+-------+--------+
|2026-05-14 00:00:00|host45|86.89  |86.89   |
|2026-05-13 23:00:00|host02|75.85  |91.69   |
|2026-05-13 23:00:00|host03|71.60  |94.64   |
|2026-05-13 23:00:00|host04|65.25  |79.62   |
|2026-05-13 23:00:00|host06|74.00  |76.69   |
|2026-05-13 23:00:00|host07|63.75  |75.88   |
|2026-05-13 23:00:00|host08|32.31  |32.31   |
|2026-05-13 23:00:00|host09|48.57  |63.31   |
|2026-05-13 23:00:00|host10|45.09  |55.19   |
|2026-05-13 23:00:00|host12|41.85  |53.83   |
|2026-05-13 23:00:00|host13|59.53  |88.23   |
|2026-05-13 23:00:00|host16|67.58  |80.03   |
|2026-05-13 23:00:00|host17|51.83  |83.22   |
|2026-05-13 23:00:00|host18|71.62  |87.75   |
|2026-05-13 23:00:00|host19|46.32  |76.70   |
|2026-05-13 23:00:00|host20|67.01  |67.01   |
|2026-05-13 23:00:00|host22|50.34  |57.67   |
|2026-05-13 23:00:00|host23|56.17  |56.17   |
|2026-05-13 23:00:00|host24|48.35 

## Spark rolling 6-hour average by host


In [6]:
hourly_base = (df_b.groupBy('host','hour_bucket').agg(F.avg('cpu_pct').alias('avg_cpu')))
w = Window.partitionBy('host').orderBy('hour_bucket').rowsBetween(-5,0)
rolling = (hourly_base.withColumn('rolling_6h_avg', F.round(F.avg('avg_cpu').over(w),2)).withColumn('avg_cpu', F.round(F.col('avg_cpu'),2)).orderBy('host','hour_bucket'))
rolling.show(50, truncate=False)


+------+-------------------+-------+--------------+
|host  |hour_bucket        |avg_cpu|rolling_6h_avg|
+------+-------------------+-------+--------------+
|host01|2026-04-30 09:00:00|44.96  |44.96         |
|host01|2026-04-30 10:00:00|51.19  |48.07         |
|host01|2026-04-30 11:00:00|47.72  |47.96         |
|host01|2026-04-30 12:00:00|62.18  |51.51         |
|host01|2026-04-30 13:00:00|55.72  |52.35         |
|host01|2026-04-30 14:00:00|52.79  |52.43         |
|host01|2026-04-30 15:00:00|53.13  |53.79         |
|host01|2026-04-30 16:00:00|63.82  |55.89         |
|host01|2026-04-30 17:00:00|52.63  |56.71         |
|host01|2026-04-30 18:00:00|71.09  |58.20         |
|host01|2026-04-30 19:00:00|75.75  |61.53         |
|host01|2026-04-30 20:00:00|45.44  |60.31         |
|host01|2026-04-30 21:00:00|47.39  |59.35         |
|host01|2026-04-30 22:00:00|76.19  |61.42         |
|host01|2026-04-30 23:00:00|65.93  |63.63         |
|host01|2026-05-01 00:00:00|71.25  |63.66         |
|host01|2026